In [0]:
%sql
-- Qual foi a receita líquida da regional Serra em julho?
SELECT f.regional,
       SUM(v.valor_total_venda) AS receita_liquida
FROM workspace.lakehouse_panvel.silver_vendas v
JOIN workspace.lakehouse_panvel.bronze_filiais f
  ON v.id_filial = f.id_filial
WHERE v.status_venda = 'Concluida'
  AND f.regional = 'Serra'
  AND date_trunc('month', v.data_venda) = date_trunc('month', DATE'2026-07-01')
GROUP BY f.regional;

In [0]:
%sql
-- Qual foi o crescimento da Panvel Porto Alegre?
WITH receita_mensal AS (
  SELECT v.id_filial, date_trunc('month', v.data_venda) AS mes,
         SUM(v.valor_total_venda) AS receita
  FROM workspace.lakehouse_panvel.silver_vendas v
  WHERE v.status_venda = 'Concluida'
  GROUP BY 1, 2
)
SELECT f.nome_filial,
       r_atual.receita  AS receita_mes_atual,
       r_anterior.receita AS receita_mes_anterior,
       (r_atual.receita - r_anterior.receita) / r_anterior.receita AS crescimento
FROM receita_mensal r_atual
JOIN receita_mensal r_anterior
  ON r_atual.id_filial = r_anterior.id_filial
  AND r_anterior.mes = add_months(r_atual.mes, -1)
JOIN workspace.lakehouse_panvel.bronze_filiais f
  ON r_atual.id_filial = f.id_filial
WHERE f.nome_filial = 'Panvel Porto Alegre'
  AND r_atual.mes = date_trunc('month', current_date());

In [0]:
%sql
-- Qual venda de produtos da categoria própria em relação a rede?
SELECT p.tipo_categoria,
       SUM(i.valor_final_item) AS receita,
       SUM(i.valor_final_item) / SUM(SUM(i.valor_final_item)) OVER () AS participacao
FROM workspace.lakehouse_panvel.silver_itens_venda i
JOIN workspace.lakehouse_panvel.silver_produtos p
  ON i.id_produto = p.id_produto
JOIN workspace.lakehouse_panvel.silver_vendas v
  ON i.id_venda = v.id_venda
WHERE v.status_venda = 'Concluida'
GROUP BY p.tipo_categoria;

In [0]:
%sql
-- Qual foi a evolução da receita da regional Metro POA?
WITH receita_mensal AS (
  SELECT f.regional, date_trunc('month', v.data_venda) AS mes,
         SUM(v.valor_total_venda) AS receita
  FROM workspace.lakehouse_panvel.silver_vendas v
  JOIN workspace.lakehouse_panvel.bronze_filiais f
    ON v.id_filial = f.id_filial
  WHERE v.status_venda = 'Concluida'
  GROUP BY 1, 2
)
SELECT r_atual.regional,
       r_atual.receita   AS receita_mes_atual,
       r_ano_anterior.receita AS receita_mesmo_mes_ano_anterior,
       (r_atual.receita - r_ano_anterior.receita) / r_ano_anterior.receita AS evolucao
FROM receita_mensal r_atual
JOIN receita_mensal r_ano_anterior
  ON r_atual.regional = r_ano_anterior.regional
  AND r_ano_anterior.mes = add_months(r_atual.mes, -12)
WHERE r_atual.regional = 'Metro POA'
  AND r_atual.mes = date_trunc('month', current_date());